In [ ]:
#%matplotlib inline
%time from hikyuu.interactive import *
use_draw_engine('matplotlib')

# I. Strategy analysis

## Original description

Position-building condition: the weekly EXPMA exp1 and exp2 golden cross upward, use B=50% of the funds to buy the stock; after the position is built successfully, the sell condition takes effect

Sell condition S1: when the daily exp1 and exp2 dead cross downward, sell S=50% of the position

Buy condition B1: when the daily exp1 and exp2 golden cross upward, buy S shares (the number of the shares sold by the sell condition S1)

S1 and B1 loop like this

The liquidation condition: when the weekly exp1 and exp2 dead cross


## Strategy analysis

Market environment: none

System valid condition: the weekly EMA1 (fast line) and EMA2 (slow line) golden cross upward until the two dead cross; when the system is valid, establish the initial position

Signal generator:
- Buy: the daily EMA1 (fast line) and EMA2 (slow line) golden cross upward
- Sell: the daily EMA1 (fast line) and EMA2 (slow line) dead cross downward

Stop-loss / take-profit: none

Money management:
- Initial position: use 50% of the funds
- Buy: 50% of the shares held at the initial position
- Sell: 50% of the shares held at the initial position

Profit goal: none

# II. Implement the system parts

## Custom system valid condition strategy

In [ ]:
def getNextWeekDateList(week):
    from datetime import timedelta
    py_week = week.datetime()
    next_week_start = py_week + timedelta(days = 7 - py_week.weekday())
    next_week_end = next_week_start + timedelta(days=5)
    return get_date_range(Datetime(next_week_start), Datetime(next_week_end))
#ds = getNextWeekDateList(Datetime(201801010000))
#for d in ds:
#    print(d)

In [ ]:
def DEMO_CN(self):
    """ The system is valid when DIF > DEA
    Parameters:
    fast_n: the weekly DIF window
    slow_n: the weekly DEA window
    """
    k = self.to
    if (len(k) <= 10):
        return
    
    #-----------------------------
    # Weekly line        
    #-----------------------------
    week_q = Query(k[0].datetime, k[-1].datetime, ktype=Query.WEEK)
    week_k = k.get_stock().get_kdata(week_q)
    
    n1 = self.get_param("week_macd_n1")
    n2 = self.get_param("week_macd_n2")
    n3 = self.get_param("week_macd_n3")
    m = MACD(CLOSE(week_k), n1, n2, n3)
    fast = m.get_result(0)
    slow = m.get_result(1)

    x = fast > slow
    for i in range(x.discard, len(x)-1):
        if (x[i] >= 1.0):
            # It needs to be extended to the daily line (must be the next week)
            date_list = getNextWeekDateList(week_k[i].datetime)
            for d in date_list:
                self._add_valid(d)

## Custom signal generator

In [ ]:
# This example does not need it; the built-in SG_Cross function can be used directly

## Custom money management strategy

In [ ]:
class DEMO_MM(MoneyManagerBase):
    """
    Initial position: use 50% of the funds
    Buy: 50% of the shares held at the initial position
    Sell: 50% of the shares held at the initial position
    """
    def __init__(self):
        super(DEMO_MM, self).__init__("MACD_MM")
        self.set_param("init_position", 0.5) # Custom initial position parameter, the percentage of the funds occupied
        self.next_buy_num = 0
        
    def _reset(self):
        self.next_buy_num = 0
        #pass
        
    def _clone(self):
        mm = DEMO_MM()
        mm.next_buy_num = self.next_buy_num
        #return DEMO_MM()
    
    def _get_buy_num(self, datetime, stk, price, risk, part_from):
        tm = self.tm
        cash = tm.current_cash
        
        # If the signal comes from the system valid condition, establish the initial position
        if part_from == System.Part.CONDITION:
            #return int((cash * 0.5 // price // stk.atom) * stk.atom)  # MoneyManagerBase already guarantees that the buy is an integer multiple of the minimum trading quantity
            self.next_buy_num = 0 # Clear the number of the shares bought/sold in a rolling way during the previous position-building period
            return int(cash * self.get_param("init_position") // price)
        
        # Not the initial position, buy the same quantity
        return self.next_buy_num
    
    def _get_sell_num(self, datetime, stk, price, risk, part_from):
        tm = self.tm
        position = tm.get_position(datetime, stk)
        current_num = int(position.number * 0.5)
        
        # Record the number of the shares at the first sell, so as to buy the same quantity next time
        if self.next_buy_num == 0:
            self.next_buy_num = current_num 
            
        return current_num # The return type must be an integer

# III. Build and run the system

## Modify and set the common parameters

Every system part and the TradeManager have their own common parameters that affect the system running; you can check the help and experiment for the details.

For example: this example currently uses the system valid condition for the initial position building, so the system common parameter cn_open_position must be set to True. Otherwise, without the initial position, there will be no subsequent sell and no trade at all.

In [ ]:
# System parameters
# delay=True #(bool): whether to delay the trade to the open of the next bar
# delay_use_current_price=True #(bool): in the case of a delayed operation, whether to calculate the new stop-loss/take-profit/target price with the price of the bar at the current trade or use the result calculated last time
# max_delay_count=3 #(int): the limit on the number of the consecutive delayed trade requests
# tp_monotonic=True #(bool): the take-profit increases monotonically
# tp_delay_n=3 #(int): the number of the days when the take-profit delay starts, i.e. the take-profit strategy takes effect only after several days of the actual trading
# ignore_sell_sg=False #(bool): ignore the sell signal, and sell only by the stop-loss/take-profit and the other ways
# ev_open_position=False #(bool): whether to use the market environment for the initial position building

cn_open_position=True #(bool): whether to use the system valid condition for the initial position building

# MoneyManager common parameters
# auto-checkin=False #(bool): when the account cash is insufficient to buy the quantity indicated by the money management strategy, automatically deposit (checkin) enough cash into the account.
# max-stock=20000 #(int): the maximum number of the kinds of the held securities (i.e. how many stocks are held, not the position size of each stock)
# disable_ev_force_clean_position=False #(bool): disable forcibly clearing the positions when the market environment becomes invalid
# disable_cn_force_clean_position=False #(bool): disable forcibly clearing the positions when the system valid condition becomes invalid


## Set the private parameters and the target to be tested

In [ ]:
# Account parameters
init_cash = 500000 # Account initial capital
init_date = '1990-1-1' # Account creation date

# Signal generator parameters
week_n1 = 12
week_n2 = 26
week_n3 = 9

# Select the target and the test range
stk = sm['sz000002']

# If it is the same K-line level, the index number can be used; if different K-line levels are used, it is recommended to use the date as the parameter
# Besides, if the data volume is too large, the matplotlib drawing will be slower
start_date = Datetime('2016-01-01')  
end_date = Datetime()

## Build the system instance

In [ ]:
# Create a simulated trading account for backtesting with an initial capital of 300,000
my_tm = crtTM(date=Datetime(init_date), init_cash = init_cash)

# Create the system instance
my_sys = SYS_Simple()

my_sys.set_param("cn_open_position", cn_open_position)

my_sys.tm = my_tm
my_cn = crtCN(DEMO_CN, 
              {'week_macd_n1': week_n1, 'week_macd_n2': week_n2, 'week_macd_n3': week_n3}, 
                'DEMO_CN')  
my_sys.cn = my_cn
my_sys.sg = SG_Cross(EMA(C, n=week_n1), EMA(C, n=week_n2))
my_mm = DEMO_MM()
my_sys.mm = my_mm

## Run the system

In [ ]:
q = Query(start_date, end_date, ktype=Query.DAY)
my_sys.run(stk, q)

# Save the trade records and positions to a temporary directory, which can be viewed with Excel
# The temporary directory is usually set to the tmp subdirectory under the data directory
# If the excel records are open, remember to close the excel file before running the system again; otherwise the new results cannot be saved
my_tm.tocsv(sm.tmpdir())

# IV. View the equity curve and the performance statistics

In [ ]:
# Draw the equity return curve
x = my_tm.get_profit_curve(stk.get_datetime_list(q), Query.DAY)
#x = my_tm.getFundsCurve(stk.getDatetimeList(q), KQuery.DAY) # Equity net value curve
PRICELIST(x).plot()

In [ ]:
# Backtest statistics
per = my_tm.get_performance()
print(per.to_df())

# V. Maybe you want to see the figure

In [ ]:
my_sys.performance()

# VI. Maybe you want to see the situation of all the stocks

In [ ]:
import pandas as pd
def calTotal(blk, q):
    per = Performance()
    s_name = []
    s_code = []
    x = []
    for stk in blk:
        my_sys.run(stk, q)
        per.statistics(my_tm, Datetime.now())
        s_name.append(stk.name)
        s_code.append(stk.market_code)
        x.append(per["Current Total Assets"])
    return pd.DataFrame({'code': s_code, 'name': s_name, 'total_assets': x})

%time data = calTotal(blocka, q)

In [ ]:
# Save to a CSV file
# data.to_csv(sm.tmpdir() + '/statistics.csv')
data[:10]